[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Smiledxd/python_data/blob/main/sesiones/S08_numpy_aleatorios_algebra.ipynb)

# Sesión 08 · Aleatorios, álgebra lineal y jefe final de NumPy

**Módulo 2: NumPy** · ⏱️ Duración estimada: 60 minutos

## 🎯 Objetivos
Al terminar esta sesión podrás:
1. Generar números aleatorios reproducibles con `default_rng` y una semilla: `random`, `integers`, `normal` y `choice`.
2. Calcular productos punto y productos de matrices con `@`, normas y sistemas de ecuaciones con `np.linalg`.
3. Medir con `%timeit` cuánto más rápida es una operación vectorizada que un bucle.
4. Analizar de principio a fin un dataset simulado de ventas usando solo NumPy.

## 📋 Qué debes saber antes
Sesiones 4 a 7: arrays 1D y 2D, máscaras, `axis` y listas de índices.

## 🧭 Cómo trabajar este notebook
- Ejecuta las celdas **en orden**, de arriba abajo, con **Shift + Enter**.
- En cada ✍️ **Tu turno** escribe tu código debajo de `# Tu código aquí`.
- Después ejecuta la celda ✅ **Verificar**. Si aparece ❌, lee el motivo, corrige y vuelve a verificar.
- Si te atascas, abre la 💡 **Pista**. Hay dos, de menos a más ayuda.
- En los ejercicios con números aleatorios, los verificadores revisan propiedades (rango, forma, promedio aproximado), no valores exactos.

## ⚙️ Setup
Ejecuta la celda siguiente **al empezar** (y otra vez si reinicias el entorno). Genera los datos de práctica y carga las funciones que revisan tus respuestas.

⚠️ **Ejecútala y no la edites.**

In [ ]:
#@title ⚙️ Setup: ejecuta esta celda y no la edites { display-mode: "form" }
# Prepara los datos de la sesión y las funciones que revisan tus respuestas.
import copy
import hashlib
import math
import statistics

import numpy as np

rng = np.random.default_rng(42)
np.set_printoptions(suppress=True)

# ---------- Datos de práctica: ventas de tiendas ----------
nombres_tiendas = np.array(["Miraflores", "Surco", "Lince", "Barranco", "San Isidro"])
productos = np.array(["polo", "jean", "casaca", "gorra"])
precios_prod = np.array([39.9, 119.9, 189.9, 25.0])
unidades_semana = rng.integers(0, 40, size=4)
unidades_tp = rng.integers(0, 60, size=(5, 4))                       # tiendas × productos
precio_costo = np.column_stack([precios_prod, np.round(precios_prod * rng.uniform(0.4, 0.7, 4), 2)])
A = np.array([[3.0, 2.0], [1.0, 4.0]])
b = np.array([250.0, 300.0])

# Transacciones simuladas para el reto (una posición = una venta)
_n = 500
tienda_idx = rng.integers(0, 5, _n)
producto_idx = rng.integers(0, 4, _n)
unidades_tx = rng.integers(1, 7, _n)
descuento_tx = rng.choice([0.0, 0.1, 0.2], _n, p=[0.6, 0.3, 0.1])
dia_tx = rng.integers(1, 31, _n)

# ---------- Datos de práctica: clientes y movimientos bancarios ----------
ids_clientes = np.arange(1001, 1051)
canales_opc = np.array(["app", "agencia", "cajero"])
cliente_a = np.round(rng.normal(0, 1, 3), 3)
cliente_b = np.round(rng.normal(0, 1, 3), 3)
montos_grandes = np.round(rng.normal(-50, 200, 100_000), 2)

_NOMBRES = ["nombres_tiendas", "productos", "precios_prod", "unidades_semana", "unidades_tp", "precio_costo",
            "A", "b", "tienda_idx", "producto_idx", "unidades_tx", "descuento_tx", "dia_tx", "ids_clientes",
            "canales_opc", "cliente_a", "cliente_b"]
_D = copy.deepcopy({k: globals()[k] for k in _NOMBRES})
# Versiones en listas de Python: los verificadores recalculan con bucles, sin NumPy.
_L = {k: (v.tolist() if isinstance(v, np.ndarray) else v) for k, v in _D.items()}

# ---------- Herramientas de verificación ----------
_FALTA = object()


def _h(valor):
    if isinstance(valor, str):
        valor = valor.strip().lower()
    return hashlib.sha256(f"{type(valor).__name__}|{valor!r}".encode("utf-8")).hexdigest()


def _cerca(a, b, tol=1e-9):
    return math.isclose(a, b, rel_tol=1e-9, abs_tol=tol)


def _dos_decimales(x):
    return abs(x * 100 - round(x * 100)) < 1e-6


def _corto(valor, n=60):
    if type(valor).__module__ == "numpy" and getattr(valor, "shape", None) == ():
        valor = valor.item()
    texto = repr(valor)
    return texto if len(texto) <= n else texto[:n] + "…"


def _igual(a, b, tol=1e-6):
    """Compara exigiendo el mismo tipo en None/bool y tolerancia en decimales."""
    if b is None or isinstance(b, bool):
        return type(a) is type(b) and a == b
    if isinstance(b, (int, float)) and not isinstance(b, bool):
        return (isinstance(a, (int, float)) and not isinstance(a, bool)
                and math.isclose(a, b, rel_tol=1e-9, abs_tol=tol))
    if isinstance(b, (list, tuple)):
        return (type(a) is type(b) and len(a) == len(b)
                and all(_igual(x, y, tol) for x, y in zip(a, b)))
    if isinstance(b, dict):
        return (isinstance(a, dict) and set(a) == set(b)
                and all(_igual(a[k], b[k], tol) for k in b))
    return type(a) is type(b) and a == b


class _Revision:
    def __init__(self, titulo):
        self.titulo = titulo
        self.errores = 0
        print(f"── {titulo} ──")

    def ok(self, msg):
        print(f"✅ {msg}")

    def mal(self, msg):
        self.errores += 1
        print(f"❌ {msg}")

    def var(self, nombre, tipo=None):
        valor = globals().get(nombre, _FALTA)
        if valor is _FALTA:
            self.mal(f"No encuentro `{nombre}`. ¿Ejecutaste tu celda? ¿Escribiste bien el nombre?")
            return _FALTA
        if tipo is not None and not (type(valor) is tipo or (isinstance(tipo, tuple) and type(valor) in tipo)):
            esperado = tipo.__name__ if not isinstance(tipo, tuple) else " o ".join(t.__name__ for t in tipo)
            self.mal(f"`{nombre}` es de tipo {type(valor).__name__} y se esperaba {esperado}.")
            return _FALTA
        return valor

    def funcion(self, nombre):
        f = self.var(nombre)
        if f is _FALTA:
            return _FALTA
        if not callable(f):
            self.mal(f"`{nombre}` existe pero no es una función. ¿La definiste con `def`?")
            return _FALTA
        return f

    def caso(self, texto, f, args=(), kwargs=None, esperado=None, igual=None, motivo="no es lo esperado", tol=0.0051):
        """Llama a f con copias de los argumentos y compara sin mostrar el valor esperado."""
        import copy
        try:
            obtenido = f(*copy.deepcopy(args), **copy.deepcopy(kwargs or {}))
        except Exception as e:
            self.mal(f"`{texto}` lanzó {type(e).__name__}: {e}")
            return False
        bien = igual(obtenido, esperado) if igual else _igual(obtenido, esperado, tol)
        if bien:
            self.ok(f"`{texto}` funciona.")
        else:
            self.mal(f"`{texto}` devolvió {_corto(obtenido)}; {motivo}.")
        return bien

    def valor(self, nombre, esperado, tipo=None, pista="revisa el cálculo", igual=None):
        v = self.var(nombre, tipo)
        if v is _FALTA:
            return
        bien = igual(v, esperado) if igual else _igual(v, esperado)
        if bien:
            self.ok(f"`{nombre}` es correcto.")
        else:
            self.mal(f"`{nombre}` vale {_corto(v)}; {pista}.")

    def texto_limpio(self, nombre, valor):
        if valor != valor.strip():
            self.mal(f"`{nombre}` tiene espacios o saltos de línea al inicio o al final: {valor!r}")
            return False
        return True

    def predicciones(self, esperados):
        for nombre, hash_ok in esperados.items():
            v = self.var(nombre)
            if v is _FALTA:
                continue
            if _h(v) == hash_ok:
                self.ok(f"`{nombre}` es correcto.")
            else:
                self.mal(f"`{nombre}` no es correcto. Razónalo otra vez y luego compruébalo ejecutando la expresión en una celda nueva.")

    def fin(self):
        if self.errores == 0:
            print(f"🎉 ¡{self.titulo} superado!")
        else:
            cuantos = "el punto marcado" if self.errores == 1 else f"los {self.errores} puntos marcados"
            print(f"🔁 Corrige {cuantos} con ❌ y vuelve a verificar.")


def _primera_diferencia(r, nombre, tuyo, esperado):
    for i, (a, b) in enumerate(zip(tuyo, esperado)):
        if a != b:
            r.mal(f"`{nombre}` no tiene el formato pedido. La diferencia empieza en el carácter {i}: "
                  f"desde ahí tu texto dice {tuyo[i:i + 15]!r}.")
            return
    n = abs(len(esperado) - len(tuyo))
    cuantos = "1 carácter" if n == 1 else f"{n} caracteres"
    if len(tuyo) < len(esperado):
        r.mal(f"`{nombre}` está incompleto: le {'falta' if n == 1 else 'faltan'} {cuantos} al final.")
    else:
        r.mal(f"`{nombre}` tiene {cuantos} de más al final: {tuyo[len(esperado):]!r}.")


def _es_numero(x):
    return isinstance(x, (int, float, np.integer, np.floating)) and not isinstance(x, (bool, np.bool_))


def _esc(r, nombre, esperado, pista, tol=1e-6):
    v = r.var(nombre)
    if v is _FALTA:
        return
    if isinstance(v, np.ndarray) and v.shape == ():
        v = v.item()
    if not _es_numero(v):
        r.mal(f"`{nombre}` es de tipo {type(v).__name__} y se esperaba un número.")
    elif abs(float(v) - esperado) <= tol:
        r.ok(f"`{nombre}` es correcto.")
    else:
        r.mal(f"`{nombre}` vale {_corto(v.item() if hasattr(v, 'item') else v)}; {pista}.")


def _arr(r, nombre, esperado, pista, tol=1e-6, tipos=None):
    v = r.var(nombre)
    if v is _FALTA:
        return
    if not isinstance(v, np.ndarray):
        r.mal(f"`{nombre}` es de tipo {type(v).__name__} y se esperaba un array de NumPy (`np.ndarray`).")
        return
    esperado = np.array(esperado)
    if v.shape != esperado.shape:
        r.mal(f"`{nombre}` tiene forma {v.shape} y se esperaba {esperado.shape}.")
        return
    if tipos and v.dtype.kind not in tipos:
        nombres = {"b": "bool", "i": "entero", "u": "entero", "f": "decimal (float)", "U": "texto"}
        r.mal(f"`{nombre}` tiene dtype {v.dtype} y se esperaba un tipo {' o '.join(sorted({nombres[t] for t in tipos}))}.")
        return
    if v.dtype.kind in "USO" or esperado.dtype.kind in "USO":
        bien = v.tolist() == esperado.tolist()
    elif v.dtype.kind == "b" or esperado.dtype.kind == "b":
        bien = np.array_equal(v, esperado)
    else:
        bien = np.allclose(v.astype(float), esperado.astype(float), rtol=0, atol=tol, equal_nan=True)
    if bien:
        r.ok(f"`{nombre}` es correcto.")
    else:
        r.mal(f"`{nombre}` tiene la forma correcta pero sus valores no coinciden; {pista}.")




def _sin_cambios(r, *nombres):
    for n in nombres:
        actual = globals().get(n)
        original = _D[n]
        if isinstance(original, np.ndarray):
            igual = isinstance(actual, np.ndarray) and actual.shape == original.shape and np.array_equal(actual, original, equal_nan=original.dtype.kind == "f")
        else:
            igual = actual == original
        if not igual:
            r.mal(f"`{n}` cambió. No modifiques los datos originales; vuelve a ejecutar el setup.")


def _array_de(r, nombre, forma, tipos):
    v = r.var(nombre)
    if v is _FALTA:
        return None
    if not isinstance(v, np.ndarray):
        r.mal(f"`{nombre}` es de tipo {type(v).__name__} y se esperaba un array de NumPy.")
        return None
    if v.shape != forma:
        r.mal(f"`{nombre}` tiene forma {v.shape} y se esperaba {forma}.")
        return None
    if v.dtype.kind not in tipos:
        r.mal(f"`{nombre}` tiene dtype {v.dtype}, que no es el tipo esperado.")
        return None
    return v


def _prueba(r, condicion, bien, mal):
    r.ok(bien) if condicion else r.mal(mal)


def check_ejercicio_1():
    r = _Revision("Ejercicio 1 · Parte A")
    g = r.var("gen")
    if g is not _FALTA:
        _prueba(r, isinstance(g, np.random.Generator), "`gen` es un generador de NumPy.",
                "`gen` debería ser el generador que devuelve `np.random.default_rng(...)`.")
    v = _array_de(r, "unidades_sim", (100,), "iu")
    if v is not None:
        if v.min() < 1 or v.max() > 10:
            r.mal(f"`unidades_sim` tiene valores entre {v.min()} y {v.max()}; deberían estar entre 1 y 10.")
        elif 10 not in v.tolist():
            r.mal("En `unidades_sim` nunca aparece el 10: recuerda que el límite superior de `integers` no se incluye.")
        else:
            r.ok("`unidades_sim` tiene 100 enteros entre 1 y 10.")
    v = _array_de(r, "uniformes", (5,), "f")
    if v is not None:
        _prueba(r, bool(((v >= 0) & (v < 1)).all()), "`uniformes` tiene 5 decimales entre 0 y 1.",
                "`uniformes` debería tener valores entre 0 (incluido) y 1 (excluido).")
    v = _array_de(r, "tickets_sim", (1000,), "f")
    if v is not None:
        media, desv = statistics.fmean(v.tolist()), statistics.pstdev(v.tolist())
        if abs(media - 80) > 2.5:
            r.mal(f"La media de `tickets_sim` es {media:.1f}; revisa el parámetro de la media.")
        elif abs(desv - 15) > 2:
            r.mal(f"La desviación de `tickets_sim` es {desv:.1f}; revisa el parámetro de la desviación.")
        else:
            r.ok("`tickets_sim` tiene la media y la desviación pedidas (aproximadamente).")
    m1, m2, m3 = (_array_de(r, n, (10,), "iu") for n in ("muestra_1", "muestra_2", "muestra_3"))
    if m1 is not None and m2 is not None and m3 is not None:
        if not all(0 <= x <= 99 for x in m1.tolist() + m3.tolist()):
            r.mal("Las muestras deberían tener enteros del 0 al 99.")
        elif m1.tolist() != m2.tolist():
            r.mal("`muestra_1` y `muestra_2` deberían ser iguales: salen de dos generadores con la misma semilla.")
        elif m1.tolist() == m3.tolist():
            r.mal("`muestra_3` debería ser distinta: sale de un generador con otra semilla.")
        else:
            r.ok("La misma semilla repite los números y otra semilla los cambia.")
    r.fin()
    r = _Revision("Ejercicio 1 · Parte B")
    r.predicciones({
        "pred_mismo": "504fd320c035cf419fcfb0f0cbe7f3b478d10f78fda40022eeb35a86b23bce08",
        "pred_max_integers": "0e939e9ca064726168df510a5b86d75df221609e471cb3d8c10019b5aff835d1",
        "pred_random_1": "98f1c15b97f0aab4934b54ee33b6a9fd148f07ee171d3618e796e5ba5b2cbf3d",
    })
    r.fin()


def check_ejercicio_2():
    r = _Revision("Ejercicio 2 · Parte A")
    prods, ids = set(_L["productos"]), _L["ids_clientes"]
    v = _array_de(r, "pedido", (20,), "U")
    if v is not None:
        _prueba(r, set(v.tolist()) <= prods, "`pedido` tiene 20 productos del catálogo.",
                "`pedido` tiene valores que no están en `productos`.")
    v = _array_de(r, "sorteo", (5,), "iu")
    if v is not None:
        if not set(v.tolist()) <= set(ids):
            r.mal("`sorteo` tiene valores que no están en `ids_clientes`.")
        elif len(set(v.tolist())) != 5:
            r.mal("En `sorteo` hay ids repetidos: un cliente no puede ganar dos veces.")
        else:
            r.ok("`sorteo` tiene 5 clientes distintos.")
    v = _array_de(r, "canales_sim", (1000,), "U")
    if v is not None:
        lista = v.tolist()
        props = {c: lista.count(c) / 1000 for c in ("app", "agencia", "cajero")}
        esperado = {"app": 0.6, "agencia": 0.3, "cajero": 0.1}
        if set(lista) - set(esperado):
            r.mal("`canales_sim` tiene valores que no están en `canales_opc`.")
        elif any(abs(props[c] - esperado[c]) > 0.05 for c in esperado):
            r.mal("Las proporciones de `canales_sim` no se parecen a las pedidas; revisa el parámetro de probabilidades y su orden.")
        else:
            r.ok("`canales_sim` respeta las probabilidades pedidas (aproximadamente).")
    v = _array_de(r, "barajado", (50,), "iu")
    if v is not None:
        if sorted(v.tolist()) != ids:
            r.mal("`barajado` debería tener exactamente los mismos ids que `ids_clientes`.")
        elif v.tolist() == ids:
            r.mal("`barajado` quedó en el mismo orden que `ids_clientes`: no se barajó.")
        else:
            r.ok("`barajado` tiene los mismos ids en otro orden.")
    _sin_cambios(r, "ids_clientes")
    r.fin()
    r = _Revision("Ejercicio 2 · Parte B")
    r.predicciones({
        "pred_sin_reemplazo": "bd1dddfaf233665e87fb493c0364dc67523ff4525fc194616e411b16fa707f7a",
    })
    r.fin()


def check_ejercicio_3():
    r = _Revision("Ejercicio 3 · Parte A")
    p, u, tp, pc = _L["precios_prod"], _L["unidades_semana"], _L["unidades_tp"], _L["precio_costo"]
    _esc(r, "ingreso_semana", math.fsum(x * y for x, y in zip(p, u)), "precio por unidades, producto a producto, y sumado", tol=1e-6)
    _arr(r, "ingresos_tienda", [math.fsum(x * y for x, y in zip(fila, p)) for fila in tp],
         "un ingreso por tienda: cada fila de `unidades_tp` combinada con `precios_prod`")
    ic = [[math.fsum(fila[k] * pc[k][j] for k in range(4)) for j in range(2)] for fila in tp]
    _arr(r, "ingreso_costo", ic, "cada tienda debería tener dos valores: ingreso y costo")
    _arr(r, "margen_tienda", [x - y for x, y in ic], "ingreso menos costo de cada tienda")
    _esc(r, "distancia", math.sqrt(math.fsum((x - y) ** 2 for x, y in zip(_L["cliente_a"], _L["cliente_b"]))),
         "la norma de la diferencia entre los dos clientes", tol=1e-9)
    x = r.var("precios_resueltos")
    if x is not _FALTA:
        if not isinstance(x, np.ndarray) or x.shape != (2,):
            r.mal("`precios_resueltos` debería ser un array con 2 valores.")
        else:
            ok = all(abs(math.fsum(a * xi for a, xi in zip(fila, x.tolist())) - bi) < 1e-6 for fila, bi in zip(_L["A"], _L["b"]))
            _prueba(r, ok, "`precios_resueltos` cumple las dos ecuaciones.",
                    "`precios_resueltos` no cumple las ecuaciones: `A @ precios_resueltos` debería dar `b`.")
    r.fin()
    r = _Revision("Ejercicio 3 · Parte B")
    r.predicciones({
        "pred_forma_matmul": "e03dd6e03740c855c4924c36d2387943231d303d8aa9c170212dae61c012a647",
        "pred_matmul_error": "bd1dddfaf233665e87fb493c0364dc67523ff4525fc194616e411b16fa707f7a",
    })
    r.fin()


def check_ejercicio_4():
    r = _Revision("Ejercicio 4")
    casos = [("np.array([-10.0, 5.0, -2.5])", np.array([-10.0, 5.0, -2.5])), ("np.array([])", np.array([])),
             ("np.array([3.0, 4.0])", np.array([3.0, 4.0])), ("np.array([-0.5])", np.array([-0.5]))]
    for nombre in ("total_con_bucle", "total_vectorizado"):
        f = r.funcion(nombre)
        if f is _FALTA:
            continue
        for texto, arr in casos:
            esperado = -math.fsum(x for x in arr.tolist() if x < 0)
            r.caso(f"{nombre}({texto})", f, (arr,), esperado=esperado,
                   igual=lambda a, b: _es_numero(a) and abs(float(a) - b) < 1e-9,
                   motivo="sin egresos debía devolver 0" if esperado == 0 else "debería ser la suma de los egresos en positivo")
    v = r.var("veces")
    if v is not _FALTA:
        if not _es_numero(v):
            r.mal("`veces` debería ser un número: tiempo del bucle entre tiempo vectorizado.")
        elif float(v) <= 1:
            r.mal(f"`veces` vale {float(v):.2f}: la versión vectorizada debería ser más rápida. ¿Dividiste en el orden correcto?")
        else:
            r.ok(f"La versión vectorizada fue unas {float(v):.0f} veces más rápida.")
    r.fin()


def _reto_ref():
    p = _L["precios_prod"]
    ti, pi, u, d, dia = (_L[k] for k in ("tienda_idx", "producto_idx", "unidades_tx", "descuento_tx", "dia_tx"))
    precio = [p[i] for i in pi]
    importe = [round(uu * pp * (1 - dd), 2) for uu, pp, dd in zip(u, precio, d)]
    total = [round(math.fsum(x for x, t in zip(importe, ti) if t == k), 2) for k in range(5)]
    matriz = [[sum(uu for uu, t, q in zip(u, ti, pi) if t == k and q == j) for j in range(4)] for k in range(5)]
    bruto = [math.fsum(fila[j] * p[j] for j in range(4)) for fila in matriz]
    diarias = [round(math.fsum(x for x, dd in zip(importe, dia) if dd == k), 2) for k in range(1, 31)]
    return precio, importe, total, matriz, bruto, diarias


def check_reto():
    r = _Revision("Reto final")
    precio, importe, total, matriz, bruto, diarias = _reto_ref()
    _arr(r, "precio_tx", precio, "usa `producto_idx` como lista de índices sobre `precios_prod`")
    _arr(r, "importe_tx", importe, "unidades por precio con el descuento aplicado, redondeado a 2 decimales", tol=0.0051)
    _arr(r, "total_por_tienda", total, "suma de `importe_tx` de cada tienda (0 a 4), con 2 decimales", tol=0.02)
    r.valor("mejor_tienda", _L["nombres_tiendas"][total.index(max(total))], None, "debería ser el nombre de la tienda con mayor total",
            igual=lambda a, b: isinstance(a, str) and str(a) == b)
    _arr(r, "matriz_tp", matriz, "unidades vendidas por tienda (filas) y producto (columnas)")
    _arr(r, "ingreso_bruto", bruto, "`matriz_tp` combinada con `precios_prod` con el operador `@`", tol=1e-6)
    _arr(r, "descuento_otorgado", [round(x - y, 2) for x, y in zip(bruto, total)],
         "ingreso bruto menos total con descuento de cada tienda", tol=0.05)
    _arr(r, "ventas_diarias", diarias, "suma de `importe_tx` de cada día, del 1 al 30", tol=0.02)
    _esc(r, "dia_pico", diarias.index(max(diarias)) + 1, "debería ser el número de día (1 a 30), no la posición")
    d = _L["descuento_tx"]
    _esc(r, "pct_con_descuento", round(len([x for x in d if x > 0]) * 100 / len(d), 1), "porcentaje de ventas con descuento mayor que 0", tol=0.051)
    _sin_cambios(r, "tienda_idx", "producto_idx", "unidades_tx", "descuento_tx", "dia_tx")
    r.fin()


def check_pro():
    r = _Revision("Nivel pro")
    _, importe, *_ = _reto_ref()
    media = statistics.fmean(importe)
    v = _array_de(r, "medias_boot", (1000,), "f")
    if v is not None:
        _prueba(r, abs(statistics.fmean(v.tolist()) - media) < 0.02 * media,
                "`medias_boot` está centrado en la media de `importe_tx`.",
                "`medias_boot` no está centrado en la media de `importe_tx`: cada remuestreo debería tener 500 ventas tomadas con reemplazo.")
    ic = r.var("ic_95")
    if ic is not _FALTA:
        ic = np.asarray(ic, dtype=float)
        if ic.shape != (2,):
            r.mal("`ic_95` debería tener 2 valores: el límite inferior y el superior.")
        elif not ic[0] < media < ic[1]:
            r.mal("`ic_95` debería contener la media de `importe_tx`.")
        elif not 0 < ic[1] - ic[0] < 0.3 * media:
            r.mal("El ancho de `ic_95` no es razonable; usa los percentiles 2.5 y 97.5 de `medias_boot`.")
        else:
            r.ok("`ic_95` es un intervalo razonable alrededor de la media.")
    r.fin()


print("✅ Setup listo. Datos generados y verificadores cargados.")

### 📦 Tus datos de hoy
Los valores se generan con una semilla fija, así que siempre salen iguales.

In [ ]:
print("🏪 Ventas de tiendas")
print("nombres_tiendas =", nombres_tiendas)
print("productos       =", productos, "| precios_prod =", precios_prod)
print("unidades_semana =", unidades_semana)
print("unidades_tp (tiendas × productos):")
print(unidades_tp)
print("precio_costo (productos × [precio, costo]):")
print(precio_costo)
print("A =", A.tolist(), "| b =", b)
print("Transacciones del reto:", tienda_idx.size, "ventas; primeras 5 →",
      "tienda", tienda_idx[:5], "producto", producto_idx[:5], "unidades", unidades_tx[:5],
      "descuento", descuento_tx[:5], "día", dia_tx[:5])
print()
print("🏦 Clientes y movimientos")
print("ids_clientes =", ids_clientes[:5], "...", ids_clientes[-1], "| canales_opc =", canales_opc)
print("cliente_a =", cliente_a, "| cliente_b =", cliente_b)
print("montos_grandes:", montos_grandes.size, "movimientos, por ejemplo", montos_grandes[:5])

---
## 1. Números aleatorios reproducibles

### 📘 Concepto
Simular datos sirve para practicar, probar ideas y estimar incertidumbre. En NumPy se trabaja con un **generador**:

```python
gen = np.random.default_rng(semilla)
```

La **semilla** fija la secuencia: dos generadores con la misma semilla producen exactamente los mismos números. Así cualquiera puede reproducir tus resultados.

| Método | Qué genera |
|---|---|
| `gen.random(n)` | `n` decimales uniformes entre 0 (incluido) y 1 (excluido) |
| `gen.integers(bajo, alto, size=n)` | enteros entre `bajo` (incluido) y `alto` (**excluido**) |
| `gen.normal(media, desv, size=n)` | valores con distribución normal |
| `gen.uniform(bajo, alto, size=n)` | decimales uniformes entre `bajo` y `alto` |

Cada llamada avanza el generador: si llamas dos veces a `gen.random()`, obtienes dos números distintos.

In [ ]:
gen_ej = np.random.default_rng(7)
print(gen_ej.random(3))
print(gen_ej.integers(1, 7, size=10))          # como tirar un dado 10 veces
print(np.round(gen_ej.normal(100, 10, size=5), 1))

otro_ej = np.random.default_rng(7)
print(otro_ej.random(3))                        # misma semilla: mismos números que la primera línea

### ✍️ Tu turno · Ejercicio 1: simular ventas
**Parte A.**
1. `gen`: un generador con semilla `2026`.
2. Con `gen`, crea `unidades_sim`: 100 enteros entre 1 y 10, **ambos incluidos**.
3. `uniformes`: 5 decimales uniformes entre 0 y 1.
4. `tickets_sim`: 1000 tickets con distribución normal de media 80 y desviación 15.
5. `muestra_1` y `muestra_2`: 10 enteros del 0 al 99, cada una con un generador **nuevo** de semilla `123`. `muestra_3`: lo mismo con semilla `124`.

**Parte B.** Predice **sin ejecutar**:

| Variable | Pregunta | Formato |
|---|---|---|
| `pred_mismo` | ¿dan el mismo número `np.random.default_rng(5).random()` y `np.random.default_rng(5).random()`? | `True` o `False` |
| `pred_max_integers` | ¿cuál es el mayor valor que puede dar `gen.integers(1, 11)`? | número |
| `pred_random_1` | ¿puede `gen.random()` devolver exactamente `1.0`? | `True` o `False` |

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_ejercicio_1()

<details><summary>💡 Pista 1</summary>

En `integers`, el segundo número es el primero que **no** puede salir. Para las muestras crea el generador en la misma línea: `np.random.default_rng(123).integers(...)`.
</details>

<details><summary>💡 Pista 2</summary>

`unidades_sim` necesita `alto = 11`. Para `tickets_sim`, los dos primeros argumentos de `normal` son la media y la desviación, y `size` la cantidad.
</details>

---
## 2. Muestreo: `choice` y `permutation`

### 📘 Concepto
- `gen.choice(opciones, size=n)` elige `n` elementos **con reemplazo**: un mismo elemento puede salir varias veces.
- Con `replace=False`, cada elemento sale como máximo una vez (como un sorteo). No puedes pedir más elementos de los que hay.
- Con `p=[...]` das la probabilidad de cada opción, en el mismo orden; las probabilidades deben sumar 1.
- `gen.permutation(a)` devuelve una **copia** barajada. En cambio, `gen.shuffle(a)` baraja el array original en su sitio.

In [ ]:
gen_ej = np.random.default_rng(1)
print(gen_ej.choice(["polo", "jean"], size=6))
print(gen_ej.choice(np.arange(1, 11), size=3, replace=False))
dados_ej = gen_ej.choice(["cara", "sello"], size=1000, p=[0.7, 0.3])
print(np.mean(dados_ej == "cara"))                  # cerca de 0.7
print(gen_ej.permutation(np.array([1, 2, 3, 4, 5])))

### ✍️ Tu turno · Ejercicio 2: pedidos, sorteos y canales
**Parte A.** Sigue usando `gen`:
1. `pedido`: 20 productos elegidos al azar de `productos` (se pueden repetir).
2. `sorteo`: 5 ganadores distintos de un sorteo entre `ids_clientes`.
3. `canales_sim`: 1000 operaciones simuladas con los canales de `canales_opc` y probabilidades 60 % app, 30 % agencia y 10 % cajero.
4. `barajado`: una copia barajada de `ids_clientes`, sin modificar el original.

**Parte B.** Predice **sin ejecutar**: `pred_sin_reemplazo` es lo que pasa con `gen.choice(productos, size=10, replace=False)`: escribe `"error"` o la forma del resultado como tupla.

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_ejercicio_2()

<details><summary>💡 Pista 1</summary>

Todo sale de `gen.choice`, salvo `barajado`. Revisa el orden de `canales_opc` antes de escribir las probabilidades.
</details>

<details><summary>💡 Pista 2</summary>

`p=[0.6, 0.3, 0.1]` va en el mismo orden que `canales_opc`. Para `barajado`, `permutation` devuelve una copia; `shuffle` cambiaría `ids_clientes`.
</details>

---
## 3. Álgebra lineal básica: `@`, normas y sistemas

### 📘 Concepto
- **Producto punto** de dos vectores del mismo largo: multiplica elemento a elemento y suma. `a @ b` o `np.dot(a, b)`. Ejemplo: precios por unidades da el ingreso total.
- **Producto de matrices** `M @ N`: cada fila de `M` hace un producto punto con cada columna de `N`. Las formas tienen que encajar por dentro: `(5, 4) @ (4, 2)` da `(5, 2)`. `(5, 4) @ (5,)` da error.
- **Norma** `np.linalg.norm(v)`: el largo del vector (raíz de la suma de cuadrados). La norma de la diferencia entre dos vectores es su **distancia**: sirve para medir qué tan parecidos son dos clientes.
- **Sistemas de ecuaciones** `np.linalg.solve(A, b)`: encuentra `x` tal que `A @ x == b`.

No confundas `*` (elemento a elemento) con `@` (producto de matrices).

In [ ]:
precios_ej = np.array([10.0, 20.0, 5.0])
cantidades_ej = np.array([3, 1, 4])
print(precios_ej * cantidades_ej, precios_ej @ cantidades_ej)

pedidos_ej = np.array([[1, 0, 2],
                       [0, 3, 1]])           # 2 clientes × 3 productos
print(pedidos_ej @ precios_ej)             # lo que paga cada cliente

print(np.linalg.norm(np.array([3.0, 4.0])))  # 5.0

# 2x + y = 7 y x + 3y = 11
print(np.linalg.solve(np.array([[2.0, 1.0], [1.0, 3.0]]), np.array([7.0, 11.0])))

### ✍️ Tu turno · Ejercicio 3: ingresos, márgenes y distancias
**Parte A.**
1. `ingreso_semana`: el ingreso total de la semana, con `precios_prod` y `unidades_semana`.
2. `ingresos_tienda`: el ingreso de cada tienda, con `unidades_tp` (tiendas × productos) y `precios_prod`.
3. `ingreso_costo`: una matriz de 5 × 2 con el ingreso y el costo de cada tienda, con `unidades_tp` y `precio_costo` (productos × [precio, costo]).
4. `margen_tienda`: ingreso menos costo de cada tienda, a partir de `ingreso_costo`.
5. `distancia`: la distancia entre `cliente_a` y `cliente_b`.
6. `precios_resueltos`: los precios `x` que cumplen el sistema `A @ x = b` (3 polos y 2 gorras cuestan 250; 1 polo y 4 gorras, 300).

**Parte B.** Predice **sin ejecutar** (una tupla o `"error"`):

| Variable | Pregunta |
|---|---|
| `pred_forma_matmul` | `(np.ones((5, 4)) @ np.ones((4, 2))).shape` |
| `pred_matmul_error` | `(np.ones((5, 4)) @ np.ones(5)).shape` |

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_ejercicio_3()

<details><summary>💡 Pista 1</summary>

Escribe las formas antes de multiplicar: `(5, 4) @ (4,)` da `(5,)` y `(5, 4) @ (4, 2)` da `(5, 2)`.
</details>

<details><summary>💡 Pista 2</summary>

`margen_tienda` es la columna 0 menos la columna 1 de `ingreso_costo`. La distancia es la norma de `cliente_a - cliente_b`.
</details>

---
## 4. Vectorizar frente a usar bucles

### 📘 Concepto
**Vectorizar** es escribir una operación sobre el array completo en lugar de recorrerlo con un bucle. NumPy ejecuta esas operaciones en código compilado, así que suelen ser decenas o cientos de veces más rápidas.

`%timeit` (un comando de Jupyter y Colab) mide cuánto tarda una línea repitiéndola varias veces. Con `-o` devuelve un objeto con el resultado; su atributo `.average` es el tiempo promedio en segundos:

```python
t = %timeit -o -n 3 -r 3 mi_funcion(datos)
t.average
```

`-n` es cuántas veces se ejecuta en cada ronda y `-r`, cuántas rondas. Valores pequeños evitan esperas largas.

In [ ]:
datos_ej = np.arange(200_000)

def suma_cuadrados_bucle(a):
    total = 0
    for x in a:
        total += x * x
    return total

def suma_cuadrados_vector(a):
    return np.sum(a * a)

print(suma_cuadrados_bucle(datos_ej) == suma_cuadrados_vector(datos_ej))
%timeit -n 3 -r 3 suma_cuadrados_bucle(datos_ej)
%timeit -n 3 -r 3 suma_cuadrados_vector(datos_ej)

### ✍️ Tu turno · Ejercicio 4: ¿cuántas veces más rápido?
1. `total_con_bucle(montos)`: la suma de los egresos (montos negativos) **en positivo**, recorriendo el array con un `for`. Si no hay egresos, devuelve 0.
2. `total_vectorizado(montos)`: lo mismo, sin bucles.
3. Mide las dos funciones con `%timeit -o -n 3 -r 3` sobre `montos_grandes` (100 000 movimientos) y guarda los resultados en `t_bucle` y `t_vector`.
4. `veces`: cuántas veces más rápida fue la versión vectorizada.

El verificador probará las funciones con arrays vacíos y sin egresos.

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_ejercicio_4()

<details><summary>💡 Pista 1</summary>

En el bucle, acumula `-x` solo cuando `x < 0`. En la versión vectorizada, filtra con una máscara y suma.
</details>

<details><summary>💡 Pista 2</summary>

La línea de medición es `t_bucle = %timeit -o -n 3 -r 3 total_con_bucle(montos_grandes)`. `veces` divide el tiempo promedio del bucle entre el de la versión vectorizada.
</details>

---
## 🏋️ Reto final (jefe de NumPy): 500 ventas simuladas
Cada posición de `tienda_idx`, `producto_idx`, `unidades_tx`, `descuento_tx` y `dia_tx` es una venta: en qué tienda (0 a 4, en el orden de `nombres_tiendas`), qué producto (0 a 3, en el orden de `productos`), cuántas unidades, qué descuento (0.1 es 10 %) y qué día del mes. Resuelve con NumPy:

1. `precio_tx`: el precio de lista de cada venta, usando `producto_idx` como lista de índices sobre `precios_prod`.
2. `importe_tx`: unidades por precio con el descuento aplicado, redondeado a 2 decimales.
3. `total_por_tienda`: el total de `importe_tx` de cada tienda (un array de 5), con 2 decimales, y `mejor_tienda`: el nombre de la tienda con mayor total.
4. `matriz_tp`: una matriz de 5 × 4 con las unidades vendidas por tienda (filas) y producto (columnas).
5. `ingreso_bruto`: el ingreso sin descuentos de cada tienda, con `matriz_tp` y `precios_prod` y el operador `@`.
6. `descuento_otorgado`: `ingreso_bruto` menos `total_por_tienda`, con 2 decimales.
7. `ventas_diarias`: el total de `importe_tx` de cada día del 1 al 30 (un array de 30), con 2 decimales, y `dia_pico`: el **número de día** con más ventas.
8. `pct_con_descuento`: el porcentaje de ventas con algún descuento, con 1 decimal.

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_reto()

<details><summary>💡 Pista 1</summary>

Para los totales por grupo, una comprensión de lista con una máscara por grupo: `np.array([importe_tx[tienda_idx == t].sum() for t in range(5)])`.
</details>

<details><summary>💡 Pista 2</summary>

`matriz_tp` necesita dos comprensiones anidadas: una por tienda y, dentro, una por producto, sumando `unidades_tx` con la máscara de las dos condiciones. Para los días, recorre `range(1, 31)`.
</details>

---
## 🚀 Nivel pro (opcional): intervalo de confianza por bootstrap
¿Qué tan seguro es el ticket promedio de `importe_tx`? El **bootstrap** lo estima remuestreando los propios datos:
1. Crea un generador con semilla 0.
2. `medias_boot`: un array de 1000 valores; cada uno es la media de una muestra de 500 elementos tomada de `importe_tx` **con reemplazo**.
3. `ic_95`: los percentiles 2.5 y 97.5 de `medias_boot` (investiga `np.percentile`). El 95 % de las medias remuestreadas cae en ese intervalo.

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_pro()

---
## ✅ Cierre: autoevaluación
Marca lo que puedes hacer sin mirar:
- [ ] Explicar para qué sirve una semilla y por qué dos generadores con la misma semilla dan lo mismo.
- [ ] Recordar que en `integers` el límite superior no se incluye.
- [ ] Simular datos con `normal`, `uniform` y `choice`, con y sin reemplazo y con probabilidades.
- [ ] Explicar la diferencia entre `permutation` y `shuffle`.
- [ ] Explicar la diferencia entre `*` y `@`, y predecir la forma de un producto de matrices.
- [ ] Calcular la distancia entre dos vectores y resolver un sistema con `np.linalg.solve`.
- [ ] Medir con `%timeit` y explicar por qué vectorizar es más rápido.
- [ ] Usar un array de índices para traer el precio de cada venta.

**Próxima sesión (S09):** empezamos pandas con `Series` y `DataFrame`.